# box-array-to-tensor-with-recipe — ex1: box raw output into MiniTensor + attach Recipe when grad-tracked

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `box-array-to-tensor-with-recipe`. Running the final beacon cell reports progress against the `Backprop: Box array as Tensor + recipe` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Box array as Tensor + recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`box-array-to-tensor-with-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "box-array-to-tensor-with-recipe"
DD_SUBTOPIC = "Backprop: Box array as Tensor + recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Box array → Tensor (with Recipe) — quick refresher

The **second half** of `wrap_forward_fn` takes a raw `torch.Tensor` (the output of `fwd_fn(*raw, **kw)`) and **boxes** it back into a `Tensor` wrapper, attaching a freshly-constructed `Recipe` so the reverse pass can find the parent edges later:

```python
out = Tensor(out_raw, requires_grad=requires_grad)
if requires_grad:
    out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
```

Two rules:
- **Always attach `out.recipe` when `requires_grad` is True.** Without   it, the reverse pass hits a non-leaf with no Recipe → KeyError in   `BACK_FUNCS.get(...)`.
- **Skip the Recipe when `requires_grad` is False.** Saves the graph   bookkeeping during inference / no_grad blocks. Leaves are also   recipe-less (`.recipe is None`).

The Recipe holds the *raw* args (already unboxed by the first half of the wrapper). The Tensor wrapper carries `.array` (the raw underlying tensor) and `.requires_grad` (propagated by the toggle/any-input gate).

### Exercise 1 — box raw output into MiniTensor + attach Recipe when grad-tracked

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the boxing half of wrap_forward_fn: wrap a raw output array in a MiniTensor, set requires_grad from the input gate, and attach a Recipe only when grad-tracked.
> Keywords: box, recipe, wrap-forward, requires-grad, leaf
> ```

**KCs targeted:** `box-array-to-tensor-with-recipe`, `recipe-dataclass`

Implement `box_with_recipe(out_raw, fwd_fn, raw_args, kwargs, parents, requires_grad)`. It is the SECOND half of `wrap_forward_fn` — given the already-computed raw output and all the bookkeeping the first half collected, produce the boxed `MiniTensor` ready to return to the caller.

Two rules:

**1. Always box.** Construct `out = MiniTensor(out_raw, requires_grad=requires_grad)`. The caller hands you the bool already computed by the requires-grad gate — don't recompute.

**2. Attach a Recipe IFF `requires_grad` is True.** When False, leave `out.recipe = None`. Two reasons: (a) inference / no_grad shouldn't pay the Recipe construction cost, (b) `out.recipe is None` is the leaf/no-graph signal the reverse pass uses to stop traversing.

The Recipe always takes the 4-tuple `(fwd_fn, raw_args, kwargs, parents)` in that order.

Inputs:
- `out_raw: torch.Tensor` — raw output of `fwd_fn(*raw_args, **kwargs)`.
- `fwd_fn: Callable` — the raw forward fn (e.g. `torch.log`).
- `raw_args: tuple` — already unboxed positional args.
- `kwargs: dict` — original keyword args.
- `parents: dict[int, MiniTensor]` — argnum → input MiniTensor.
- `requires_grad: bool` — precomputed gate result.

Returns: a `MiniTensor`.

In [ ]:
def box_with_recipe(
    out_raw,
    fwd_fn,
    raw_args: tuple,
    kwargs: dict,
    parents: dict,
    requires_grad: bool,
) -> MiniTensor:
    # Always box — even when not grad-tracked, callers expect a MiniTensor.
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    # Attach Recipe ONLY when grad-tracking — saves bookkeeping in no_grad.
    if requires_grad:
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return out


<details><summary>Solution</summary>

```python
def box_with_recipe(
    out_raw,
    fwd_fn,
    raw_args: tuple,
    kwargs: dict,
    parents: dict,
    requires_grad: bool,
) -> MiniTensor:
    # Always box — even when not grad-tracked, callers expect a MiniTensor.
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    # Attach Recipe ONLY when grad-tracking — saves bookkeeping in no_grad.
    if requires_grad:
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return out
```

**Why the conditional Recipe.** During inference, gradient information is not needed — every node would carry a Recipe that nothing ever reads. Skipping the construction saves both allocation and reference-keeping on the parents (so they can be garbage-collected sooner).

**`out.array is raw_out` (identity).** Boxing wraps; it does NOT copy. The Tensor wrapper is a thin shell — the same underlying raw tensor flows through forward and reverse passes. This is critical for cached-value reuse in backward fns (the `out` parameter of `sigmoid_back`, etc.).

**Recipe field order matters.** Recipe is constructed positionally `Recipe(fwd_fn, raw_args, kwargs, parents)`. If you accidentally build it as `Recipe(fwd_fn, kwargs, raw_args, parents)` because your IDE auto-completed wrong, the reverse pass replays `fwd_fn(*kwargs, **raw_args)` and crashes. The Recipe schema is a fixed 4-tuple — every wrapper in the codebase reads it the same way.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()